# 05 · Redes neuronales desde cero: perceptrón, MLP y retropropagación

**Módulo 5 · Sesión 13** — Deep learning (puente)

## Objetivos

Una red neuronal es una regresión logística con capas intermedias, entrenada con el
mismo descenso del gradiente de los módulos 1, 3 y 4. Lo único nuevo es cómo se calcula
el gradiente cuando hay capas: la **retropropagación**, que es la regla de la cadena de
la S3 aplicada con orden. Este notebook la construye a mano y la verifica dos veces:

1. Ver el límite del **perceptrón** (y de la regresión logística): fronteras lineales.
2. Implementar un **MLP de una capa oculta** en NumPy —propagación hacia adelante,
   pérdida y retropropagación— y verificar el gradiente contra **diferencias finitas** y
   contra el **autograd de PyTorch**.
3. Entrenar con el bucle de descenso del gradiente de siempre y ver cómo la frontera
   deja de ser recta a medida que crece la capa oculta.
4. Medir qué hacen las **funciones de activación**: por qué la sigmoide "apaga" el
   gradiente en redes profundas y ReLU no.
5. Ver la **aproximación universal** en acción (ajustar $\sin(2\pi x)$) y el sobreajuste
   que viene con ella.

La teoría está en `04-redes-neuronales.md`. El notebook 06 hace todo esto con PyTorch
sobre datos reales.

**Paquetes:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`, `torch`.

In [ ]:
# Arranque para Google Colab (en local no hace nada): trae el repositorio para que
# ../datos y ../src existan. Ejecútala antes que cualquier otra celda.
import sys
if "google.colab" in sys.modules:
    !git clone -q --depth 1 https://github.com/delany-ramirez/machine_learning /content/machine_learning
    %cd /content/machine_learning/modulo-5-no-supervisado-deep-learning/notebooks

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

SEMILLA = 42
rng = np.random.default_rng(SEMILLA)
torch.manual_seed(SEMILLA)

## 1. El perceptrón y su límite

El **perceptrón** (Rosenblatt, 1958) es la neurona original: una combinación lineal de las
entradas y un umbral, $\hat{y} = \mathbb{1}[\mathbf{w}^\top \mathbf{x} + b > 0]$. La
regresión logística de la S9 es lo mismo con una sigmoide en vez del escalón, y por eso
se entrena con gradiente. Ambos comparten la limitación que hundió al perceptrón en 1969
(Minsky y Papert): la frontera es un **hiperplano**. El ejemplo clásico es XOR; el
nuestro, las medias lunas del módulo 4.

In [ ]:
X, y = make_moons(n_samples=600, noise=0.2, random_state=SEMILLA)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=SEMILLA)


def graficar_frontera(predecir_proba, X, y, eje, titulo):
    g1, g2 = np.meshgrid(np.linspace(-1.8, 2.8, 250), np.linspace(-1.3, 1.8, 250))
    p = predecir_proba(np.c_[g1.ravel(), g2.ravel()]).reshape(g1.shape)
    eje.contourf(g1, g2, p, levels=np.linspace(0, 1, 11), cmap="RdBu_r", alpha=0.6)
    eje.contour(g1, g2, p, levels=[0.5], colors="black", linewidths=1.5)
    eje.scatter(X[:, 0], X[:, 1], c=y, cmap="RdBu_r", s=8, edgecolor="k", linewidth=0.2)
    eje.set_title(titulo)
    eje.set_xticks([])
    eje.set_yticks([])


logistica = LogisticRegression().fit(X_train, y_train)
fig, eje = plt.subplots(figsize=(6, 4.5))
graficar_frontera(lambda Z: logistica.predict_proba(Z)[:, 1], X_test, y_test, eje,
                  f"Regresión logística · accuracy de prueba {logistica.score(X_test, y_test):.3f}")
plt.show()

Una recta no puede separar dos lunas: 0.86 es lo mejor que se puede hacer con un
hiperplano. El módulo 4 resolvió esto con árboles (escalones) o con SVM (un kernel que
transforma las variables **a mano**). Una red neuronal lo resuelve **aprendiendo la
transformación**.

## 2. Un MLP de una capa oculta

Un **perceptrón multicapa** (MLP) apila capas. Con una capa oculta de $h$ neuronas:

$$
\mathbf{Z}_1 = \mathbf{X}\mathbf{W}_1 + \mathbf{b}_1 \qquad
\mathbf{H} = \phi(\mathbf{Z}_1) \qquad
\mathbf{z}_2 = \mathbf{H}\mathbf{w}_2 + b_2 \qquad
\hat{\mathbf{p}} = \sigma(\mathbf{z}_2)
$$

donde $\mathbf{W}_1 \in \mathbb{R}^{p \times h}$, $\mathbf{w}_2 \in \mathbb{R}^{h}$,
$\phi$ es la **activación** (aquí $\tanh$; ReLU más adelante) y $\sigma$ la sigmoide.
La última capa es exactamente una regresión logística — sobre $\mathbf{H}$, las $h$
variables **que la red construye**, en vez de sobre $\mathbf{X}$. Sin $\phi$, la
composición de dos capas lineales sería otra capa lineal y no se ganaría nada: la
no linealidad es lo que permite que la frontera se curve.

La pérdida es la entropía cruzada de la S9, $\mathcal{L} = -\frac{1}{n}\sum_i [y_i \log
\hat p_i + (1-y_i)\log(1-\hat p_i)]$.

In [ ]:
def sigmoide(z):
    return 1 / (1 + np.exp(-z))


def inicializar(p, h, rng, escala=0.5):
    return {"W1": rng.normal(0, escala, (p, h)), "b1": np.zeros(h),
            "w2": rng.normal(0, escala, h), "b2": 0.0}


def adelante(params, X, activacion=np.tanh):
    Z1 = X @ params["W1"] + params["b1"]
    H = activacion(Z1)
    z2 = H @ params["w2"] + params["b2"]
    p_hat = sigmoide(z2)
    return p_hat, {"X": X, "Z1": Z1, "H": H, "p_hat": p_hat}     # la caché sirve para el paso atrás


def perdida(p_hat, y):
    eps = 1e-12
    return -np.mean(y * np.log(p_hat + eps) + (1 - y) * np.log(1 - p_hat + eps))


params = inicializar(p=2, h=8, rng=rng)
p_hat, cache = adelante(params, X_train)
print(f"Pérdida inicial (pesos al azar): {perdida(p_hat, y_train):.4f}   (predecir 0.5 siempre daría {np.log(2):.4f})")

## 3. Retropropagación: la regla de la cadena, en orden

Hay que calcular $\partial \mathcal{L} / \partial \theta$ para cada parámetro. La regla
de la cadena lo hace **de atrás hacia adelante**, reutilizando en cada capa lo calculado
en la siguiente. Con $\delta_2 = \hat{\mathbf{p}} - \mathbf{y}$ (el mismo "residual" de la
regresión logística, S9) y $\odot$ el producto elemento a elemento:

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{w}_2} = \frac{1}{n}\mathbf{H}^\top \boldsymbol{\delta}_2, \qquad
\frac{\partial \mathcal{L}}{\partial b_2} = \frac{1}{n}\sum_i \delta_{2,i}
$$

$$
\boldsymbol{\Delta}_1 = (\boldsymbol{\delta}_2 \mathbf{w}_2^\top) \odot \phi'(\mathbf{Z}_1), \qquad
\frac{\partial \mathcal{L}}{\partial \mathbf{W}_1} = \frac{1}{n}\mathbf{X}^\top \boldsymbol{\Delta}_1, \qquad
\frac{\partial \mathcal{L}}{\partial \mathbf{b}_1} = \frac{1}{n}\sum_i \boldsymbol{\Delta}_{1,i}
$$

La primera línea es el gradiente de la regresión logística (módulo 4, notebook 01) con
$\mathbf{H}$ en el lugar de $\mathbf{X}$. La segunda **propaga** el error hacia atrás:
cada neurona oculta recibe la parte del error que le corresponde según su peso
$w_{2,j}$, multiplicada por la pendiente de su activación en el punto donde estaba. Para
$\tanh$, $\phi'(z) = 1 - \tanh^2(z)$.

In [ ]:
def atras(params, cache, y, derivada_activacion=lambda Z1, H: 1 - H**2):
    n = len(y)
    delta2 = cache["p_hat"] - y                                   # n
    grad = {"w2": cache["H"].T @ delta2 / n, "b2": delta2.mean()}
    Delta1 = np.outer(delta2, params["w2"]) * derivada_activacion(cache["Z1"], cache["H"])   # n × h
    grad["W1"] = cache["X"].T @ Delta1 / n
    grad["b1"] = Delta1.mean(axis=0)
    return grad


grad = atras(params, cache, y_train)

### Verificación 1: diferencias finitas

Un gradiente escrito a mano se verifica siempre. Para cada parámetro $\theta_k$, la
derivada numérica $[\mathcal{L}(\theta_k + \epsilon) - \mathcal{L}(\theta_k - \epsilon)] /
2\epsilon$ debe coincidir con la analítica. Es lento (una pasada hacia adelante por
parámetro) pero infalible, y es lo que se hace antes de confiar en cualquier
implementación.

In [ ]:
def gradiente_numerico(params, X, y, eps=1e-5):
    num = {}
    for nombre, valor in params.items():
        valor = np.atleast_1d(np.asarray(valor, dtype=float))
        g = np.zeros_like(valor)
        for idx in np.ndindex(valor.shape):
            original = valor[idx]
            valor[idx] = original + eps
            mas = perdida(adelante({**params, nombre: valor.reshape(np.shape(params[nombre]))}, X)[0], y)
            valor[idx] = original - eps
            menos = perdida(adelante({**params, nombre: valor.reshape(np.shape(params[nombre]))}, X)[0], y)
            valor[idx] = original
            g[idx] = (mas - menos) / (2 * eps)
        num[nombre] = g.reshape(np.shape(params[nombre]))
    return num


num = gradiente_numerico(params, X_train, y_train)
for nombre in params:
    analitico, numerico = np.atleast_1d(grad[nombre]), np.atleast_1d(num[nombre])
    error_rel = np.abs(analitico - numerico).max() / (np.abs(analitico).max() + 1e-12)
    print(f"{nombre}: error relativo máximo entre gradiente analítico y numérico = {error_rel:.2e}")

### Verificación 2: el autograd de PyTorch

PyTorch calcula gradientes por **diferenciación automática**: registra cada operación
de la pasada hacia adelante y aplica la regla de la cadena hacia atrás, exactamente lo
que hicimos a mano pero para cualquier grafo de operaciones. Con los mismos pesos y los
mismos datos, debe dar los mismos números.

In [ ]:
t = {k: torch.tensor(np.atleast_1d(v), dtype=torch.float64, requires_grad=True) for k, v in params.items()}
X_t, y_t = torch.tensor(X_train), torch.tensor(y_train, dtype=torch.float64)
H_t = torch.tanh(X_t @ t["W1"] + t["b1"])
z2_t = H_t @ t["w2"] + t["b2"]
perdida_t = torch.nn.functional.binary_cross_entropy_with_logits(z2_t, y_t)
perdida_t.backward()
for nombre in params:
    coincide = np.allclose(t[nombre].grad.numpy().reshape(np.shape(grad[nombre])), grad[nombre], atol=1e-10)
    print(f"{nombre}: gradiente a mano == autograd de PyTorch: {coincide}")

Las dos verificaciones coinciden a $10^{-10}$. A partir de aquí, cada vez que el notebook
06 llame a `loss.backward()`, es esto.

## 4. Entrenar: el mismo bucle de siempre

Descenso del gradiente por mini-lotes (módulo 3), sin cambiar nada más que la función
que calcula el gradiente. Comparamos capas ocultas de 1, 2, 4 y 16 neuronas.

In [ ]:
def entrenar(X, y, h, epocas=400, tasa=0.5, lote=32, rng=None, activacion=np.tanh,
             derivada=lambda Z1, H: 1 - H**2, X_val=None, y_val=None):
    rng = rng or np.random.default_rng(SEMILLA)
    params = inicializar(X.shape[1], h, rng)
    historial = {"train": [], "val": []}
    for _ in range(epocas):
        for idx in np.array_split(rng.permutation(len(X)), max(1, len(X) // lote)):
            p_hat, cache = adelante(params, X[idx], activacion)
            grad = atras(params, cache, y[idx], derivada)
            for k in params:
                params[k] = params[k] - tasa * grad[k]
        historial["train"].append(perdida(adelante(params, X, activacion)[0], y))
        if X_val is not None:
            historial["val"].append(perdida(adelante(params, X_val, activacion)[0], y_val))
    return params, historial


fig, ejes = plt.subplots(1, 4, figsize=(20, 4.3))
filas = []
for eje, h in zip(ejes, [1, 2, 4, 16]):
    params_h, hist = entrenar(X_train, y_train, h)
    acc = np.mean((adelante(params_h, X_test)[0] > 0.5) == y_test)
    filas.append({"neuronas ocultas": h, "pérdida final (train)": hist["train"][-1], "accuracy (test)": acc})
    graficar_frontera(lambda Z, p=params_h: adelante(p, Z)[0], X_test, y_test, eje, f"h = {h} · accuracy {acc:.3f}")
plt.show()
print(pd.DataFrame(filas).round(3).to_string(index=False))

Con **una** neurona oculta la frontera sigue siendo una recta (una neurona $\tanh$ es
monótona: solo puede reordenar una proyección lineal). Con dos, la frontera se quiebra
una vez; con cuatro, sigue la curva de las lunas; con dieciséis, la sigue mejor y
empieza a dibujar detalles que son ruido. Cada neurona oculta es una "recta suavizada",
y la capa de salida las combina — es el mismo principio de los ensambles (S10–S11):
modelos simples combinados hacen uno complejo.

## 5. Funciones de activación y el gradiente que se apaga

La activación $\phi$ decide dos cosas: qué forma tiene cada "pieza" de la frontera, y
cuánto gradiente pasa hacia atrás por la neurona (el factor $\phi'(\mathbf{Z}_1)$ de la
retropropagación). Tres opciones:

In [ ]:
z = np.linspace(-5, 5, 400)
activaciones = {"sigmoide": (sigmoide(z), sigmoide(z) * (1 - sigmoide(z))),
                "tanh": (np.tanh(z), 1 - np.tanh(z) ** 2),
                "ReLU": (np.maximum(0, z), (z > 0).astype(float))}
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
for nombre, (f, df) in activaciones.items():
    ejes[0].plot(z, f, label=nombre)
    ejes[1].plot(z, df, label=nombre)
ejes[0].set_title("Activación φ(z)")
ejes[1].set_title("Derivada φ'(z): cuánto gradiente deja pasar")
for eje in ejes:
    eje.legend()
    eje.set_xlabel("z")
plt.show()

La derivada de la sigmoide vale como máximo 0.25 y es casi cero fuera de $[-4, 4]$; la
de $\tanh$, como máximo 1 y también se apaga; la de ReLU vale exactamente 1 para $z > 0$
y 0 para $z < 0$. En una red de **muchas capas**, el gradiente de la primera capa es un
producto de tantos factores $\phi'$ como capas: con la sigmoide, $0.25^{L}$ en el mejor
de los casos. Es el **desvanecimiento del gradiente** (*vanishing gradient*), y la razón
de que las redes profundas no se pudieran entrenar hasta que ReLU (y otras cosas) lo
arreglaron. Lo medimos: una red de 10 capas ocultas, cada una con la inicialización
recomendada para su activación (Glorot/Xavier para sigmoide y tanh, He para ReLU), y la
norma del gradiente en cada capa, con PyTorch para no escribir la retropropagación de
$L$ capas a mano.

In [ ]:
def normas_por_capa(activacion, inicializar_pesos, capas=10, ancho=32):
    torch.manual_seed(SEMILLA)
    modulos = []
    d = 2
    for _ in range(capas):
        lineal = torch.nn.Linear(d, ancho)
        inicializar_pesos(lineal.weight)          # la inicialización recomendada para cada activación
        torch.nn.init.zeros_(lineal.bias)
        modulos += [lineal, activacion()]
        d = ancho
    modulos.append(torch.nn.Linear(d, 1))
    red = torch.nn.Sequential(*modulos)
    salida = red(torch.tensor(X_train, dtype=torch.float32)).squeeze(1)
    torch.nn.functional.binary_cross_entropy_with_logits(salida, torch.tensor(y_train, dtype=torch.float32)).backward()
    return [m.weight.grad.norm().item() for m in red if isinstance(m, torch.nn.Linear)]


fig, eje = plt.subplots(figsize=(8, 4.5))
xavier = torch.nn.init.xavier_normal_
he = lambda w: torch.nn.init.kaiming_normal_(w, nonlinearity="relu")
for nombre, act, init in [("sigmoide", torch.nn.Sigmoid, xavier), ("tanh", torch.nn.Tanh, xavier), ("ReLU", torch.nn.ReLU, he)]:
    normas = normas_por_capa(act, init)
    eje.semilogy(range(1, len(normas) + 1), normas, "o-", label=nombre)
    print(f"{nombre:<9} norma del gradiente: capa 1 = {normas[0]:.2e}   capa 10 = {normas[9]:.2e}   "
          f"cociente capa 1 / capa 10 = {normas[0] / normas[9]:.1e}")
eje.set_xlabel("Capa (1 = la más cercana a la entrada)")
eje.set_ylabel("‖∂L/∂W‖ (escala log)")
eje.set_title("Red de 10 capas ocultas en la inicialización: cuánto gradiente llega a cada capa")
eje.legend()
plt.show()

Con la sigmoide, la primera capa recibe un gradiente **seis órdenes de magnitud** menor
que la última: no aprende. Con tanh y ReLU (bien inicializadas), todas las capas reciben
gradientes del mismo orden (cociente 0.3–0.4). Por eso ReLU es la activación por defecto en las capas ocultas desde 2012, y la sigmoide
queda solo en la salida (donde sí hace falta una probabilidad). ReLU tiene su propio
problema —una neurona con $z < 0$ para todos los datos tiene gradiente cero y "muere"—,
que sus variantes (Leaky ReLU, GELU) suavizan.

## 6. Aproximación universal, y su precio

El **teorema de aproximación universal** (Cybenko, 1989; Hornik, 1991): un MLP con una
capa oculta suficientemente ancha puede aproximar cualquier función continua en un
compacto con la precisión que se quiera. Es un resultado de existencia —no dice cuántas
neuronas ni cómo encontrarlas—, pero se ve. Regresión (salida lineal, pérdida cuadrática)
sobre $y = \sin(2\pi x) + \varepsilon$, la función del módulo 3, notebook 05:

In [ ]:
def entrenar_regresion(x, y, h, epocas=3000, tasa=0.05, rng=None):
    rng = rng or np.random.default_rng(SEMILLA)
    W1, b1 = rng.normal(0, 1, (1, h)), rng.uniform(-1, 1, h)
    w2, b2 = rng.normal(0, 0.5, h) / np.sqrt(h), 0.0        # escala 1/√h: la salida no crece con el ancho
    for _ in range(epocas):                                  # descenso del gradiente por lotes completos
        Z1 = x @ W1 + b1
        H = np.tanh(Z1)
        y_hat = H @ w2 + b2
        d = 2 * (y_hat - y) / len(y)
        gw2, gb2 = H.T @ d, d.sum()
        D1 = np.outer(d, w2) * (1 - H**2)
        gW1, gb1 = x.T @ D1, D1.sum(axis=0)
        W1, b1, w2, b2 = W1 - tasa * gW1, b1 - tasa * gb1, w2 - tasa * gw2, b2 - tasa * gb2
    return lambda x_nuevo: np.tanh(x_nuevo @ W1 + b1) @ w2 + b2


x_reg = rng.uniform(0, 1, (60, 1))
y_reg = np.sin(2 * np.pi * x_reg[:, 0]) + rng.normal(0, 0.25, 60)
malla = np.linspace(0, 1, 300)[:, None]

fig, ejes = plt.subplots(1, 4, figsize=(20, 4))
for eje, h in zip(ejes, [1, 3, 10, 100]):
    f = entrenar_regresion(x_reg, y_reg, h)
    ecm_train = np.mean((f(x_reg) - y_reg) ** 2)
    ecm_verdad = np.mean((f(malla) - np.sin(2 * np.pi * malla[:, 0])) ** 2)
    eje.scatter(x_reg, y_reg, s=12, color="gray", label="datos")
    eje.plot(malla, np.sin(2 * np.pi * malla), "--", color="black", label="sin(2πx)")
    eje.plot(malla, f(malla), color="C3", lw=2, label="MLP")
    eje.set_title(f"h = {h}\nECM sobre los datos {ecm_train:.3f} · contra la verdad {ecm_verdad:.3f}")
    eje.set_ylim(-1.8, 1.8)
ejes[0].legend()
plt.show()

Con 1 neurona, una sigmoide; con 3, casi lo mismo (con pocas neuronas la optimización
se atasca en una solución pobre); con 10, el seno. Y con 100, **mejor todavía** (ECM
contra la función verdadera 0.011 frente a 0.079 con 10): el
sobreajuste que la intuición de la S8 predice para un modelo con 300 parámetros y 60
datos no aparece. Con descenso del gradiente desde pesos pequeños, una red ancha se
comporta mejor de lo que su número de parámetros sugiere — es un fenómeno real y
activo en la investigación (*sobreparametrización benigna*, doble descenso). No es una
garantía: el sobreajuste de una red depende de los datos, las épocas y la
inicialización, y aparece con pocos datos y entrenamientos largos. Sobre 120 puntos de
las lunas, con 64 neuronas:

In [ ]:
X_pocos, y_pocos = make_moons(n_samples=120, noise=0.3, random_state=SEMILLA)
X_val, y_val = make_moons(n_samples=1000, noise=0.3, random_state=SEMILLA + 1)
_, hist = entrenar(X_pocos, y_pocos, h=64, epocas=1500, tasa=0.3, lote=16, X_val=X_val, y_val=y_val)

fig, eje = plt.subplots(figsize=(8, 4))
eje.plot(hist["train"], label="pérdida de entrenamiento (120 puntos)")
eje.plot(hist["val"], label="pérdida de validación (1000 puntos nuevos)")
mejor = int(np.argmin(hist["val"]))
eje.axvline(mejor, color="gray", ls="--", label=f"mínimo de validación: época {mejor}")
eje.set_xlabel("época")
eje.set_ylabel("entropía cruzada")
eje.set_title("MLP de 64 neuronas sobre 120 puntos: la validación sube mientras el entrenamiento baja")
eje.legend()
plt.show()
print(f"Pérdida de validación mínima {min(hist['val']):.3f} en la época {mejor}; al final (época 1500): {hist['val'][-1]:.3f}")

Aquí sí: la pérdida de entrenamiento baja sin parar y la de validación toca fondo en la
época ≈120 y sube después. Es la curva en U de la S8 con las épocas en el eje $x$. Por eso
el entrenamiento de una red se vigila **siempre** con un conjunto de validación y se
detiene antes de tiempo (*early stopping*), y por eso se regulariza (*weight decay*,
*dropout*). El notebook 06 lo hace con PyTorch.

## Resumen

| Pregunta | Respuesta medida |
|---|---|
| ¿Qué añade la capa oculta? | Variables aprendidas: la logística pasa de 0.86 (recta) a ≈0.95 sobre las lunas con 4–16 neuronas |
| ¿Qué es la retropropagación? | La regla de la cadena de atrás hacia adelante; el gradiente a mano coincide con diferencias finitas y con el autograd de PyTorch a $10^{-10}$ |
| ¿Qué cambia respecto a los módulos 3 y 4? | Nada en el bucle de entrenamiento: solo la función que calcula el gradiente |
| ¿Por qué ReLU? | En 10 capas, la sigmoide deja a la primera capa con un gradiente $10^6$ veces menor que a la última; tanh y ReLU bien inicializadas, del mismo orden |
| ¿Aproxima cualquier función? | Sí: 10 neuronas ajustan el seno y 100 lo ajustan mejor, no peor. El sobreajuste aparece con pocos datos y muchas épocas (lunas: la validación sube de 0.22 a 0.29), de ahí validación, early stopping y regularización |